In [6]:
# Prerequisite
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

train_dataloader = DataLoader(training_data, batch_size=64)
test_dataloader = DataLoader(test_data, batch_size=64)

class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork()

In [7]:
# Hyperparameters
# Hyperparameters are adjustable parameters that let you control the model optimization process.
epochs = 5 # the number of times to iterator over the dataset
batch_size = 64 # the number of data samples propagated through the network before the parameters are updated
learning_rate = 1e-3 # how much to update model parameters at each batch/epoch

In [8]:
# Optimization loop
# Each epoch consists 2 major parts:
# The Train Loop
#     iterate over the training dataset and try to converge the optimal parameters
# The Validation/Test Loop
#     iterator over the test dataset to check if model is improving

# Loss Function
# Loss function measures the degree of dissimilarity of obtained result to the target value, 
# and it is the loss function that we want to minimize during training.
loss_fn = nn.CrossEntropyLoss()

# Optimizer
# Optimization is the process of adjusting model parameters to reduce model error in each training step. 
# Optimization algorithms define how this process is performed 
# (in this example we use Stochastic Gradient Descent). 
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

# Inside the training loop, optimization happens in 3 steps:
# Call optimizer.zero_grad() to reset gradients of the model parameters
# Backpropagate the prediction loss with loss.backward(), computing grad for each parameter
# Call optimzer.step() to adjust the parameters by the gradients

In [9]:
# Full implementation
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    # Set the model to training mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.train()
    for batch, (X,y) in enumerate(dataloader):
        # Compute prediction and loss
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagate
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f} [{current:>5d}/{size:>5d}]")

def test_loop(dataloader, model, loss_fn):
    # Set the model to evaluation mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    # valuating the model with torch.no_grad() ensures that no gradients are computed during test mode
    # also serves to reduce unnecessary gradient computations and memory usage for tensors with requires_grad=True
    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            test_loss += loss_fn(pred, y)
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [10]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

epochs = 10
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 2.313699 [   64/60000]
loss: 2.299027 [ 6464/60000]
loss: 2.282813 [12864/60000]
loss: 2.264946 [19264/60000]
loss: 2.252968 [25664/60000]
loss: 2.221044 [32064/60000]
loss: 2.226113 [38464/60000]
loss: 2.191847 [44864/60000]
loss: 2.174403 [51264/60000]
loss: 2.149848 [57664/60000]
Test Error: 
 Accuracy: 50.3%, Avg loss: 2.146377 

Epoch 2
-------------------------------
loss: 2.161138 [   64/60000]
loss: 2.150260 [ 6464/60000]
loss: 2.093288 [12864/60000]
loss: 2.100768 [19264/60000]
loss: 2.054929 [25664/60000]
loss: 1.985761 [32064/60000]
loss: 2.012563 [38464/60000]
loss: 1.929286 [44864/60000]
loss: 1.921722 [51264/60000]
loss: 1.858381 [57664/60000]
Test Error: 
 Accuracy: 59.6%, Avg loss: 1.860384 

Epoch 3
-------------------------------
loss: 1.894597 [   64/60000]
loss: 1.867827 [ 6464/60000]
loss: 1.748872 [12864/60000]
loss: 1.787157 [19264/60000]
loss: 1.684983 [25664/60000]
loss: 1.625836 [32064/60000]
loss: 1.647250 [38464/

In [ ]:
# Save and load model weights
import torchvision.models as models

# PyTorch models store the learned parameters in an internal state dictionary, called state_dict. 
# These can be persisted via the torch.save method
model = models.vgg16(weights='IMAGENET1K_V1')
torch.save(model.state_dict(), 'model_weights.pth')

# To load model weights, you need to create an instance of the same model first, 
# and then load the parameters using load_state_dict() method.
# Set weights_only=True to limit the functions executed during unpickling 
# to only those necessary for loading weights. 
# Using weights_only=True is considered a best practice when loading weights.
model = models.vgg16() # we do not specify ``weights``, i.e. create untrained model
model.load_state_dict(torch.load('model_weights.pth', weights_only=True))
model.eval()

# Save model weights + shapes
torch.save(model, 'model.pth')
# Load model weights and shapes
model = torch.load('model.pth', weights_only=False)